# 1. Pengaturan Path dan Memuat Dataset

In [2]:
import os, sys, subprocess
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr

REPO_URL = "https://github.com/midnightbluee2/TBP-PCD.git"

if "google.colab" in sys.modules:
    BASE = Path("/content/TBP-PCD")
    if not BASE.exists():
        subprocess.run(["git", "clone", REPO_URL, str(BASE)], check=True)
else:
    BASE = Path.cwd()

FOTO_DIR    = BASE / "data" / "foto"
HASIL_DIR   = BASE / "hasil"
LSB_DIR     = HASIL_DIR / "lsb"
LSB_CA_DIR  = HASIL_DIR / "lsb_ca"
DCT_DIR     = HASIL_DIR / "dct"
KOMPRES_DIR = HASIL_DIR / "kompresi_jpeg"

for d in (LSB_DIR, LSB_CA_DIR, DCT_DIR, KOMPRES_DIR):
    d.mkdir(parents=True, exist_ok=True)

URUTAN   = ["BACTERIA-1157929-0001", "BACTERIA-4269599-0003",
            "NORMAL-528189-0001", "NORMAL-1101909-0001"]
PESAN    = ["RM001", "RM002", "RM003", "RM004"]
UKURAN   = 512
EKSTENSI = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def tebak_label(nama):
    t = nama.lower()
    if any(k in t for k in ("pneumonia", "bacteria", "virus")):
        return "PNEUMONIA"
    return "NORMAL" if "normal" in t else "TIDAK_DIKETAHUI"

def muat_foto(folder, urutan, pesan, ukuran=512):
    jalur = [os.path.join(a, f) for a, _, fs in os.walk(folder) for f in fs
             if f.lower().endswith(EKSTENSI)]
    if not jalur:
        raise FileNotFoundError(f"Tidak ada foto di folder: {folder}")
    peta = {os.path.splitext(os.path.basename(p))[0].lower(): p for p in jalur}
    hilang = [n for n in urutan if n.lower() not in peta]
    if hilang:
        raise FileNotFoundError(f"Foto tidak ditemukan: {hilang}")
    dataset = []
    for n, ps in zip(urutan, pesan):
        img = cv2.resize(cv2.imread(peta[n.lower()], cv2.IMREAD_GRAYSCALE),
                         (ukuran, ukuran), interpolation=cv2.INTER_AREA)
        dataset.append({"nama": n, "label": tebak_label(n), "pesan": ps, "img": img})
    return dataset

data = muat_foto(str(FOTO_DIR), URUTAN, PESAN, UKURAN)
print(f"Berhasil memuat {len(data)} foto dari {FOTO_DIR}")

Berhasil memuat 4 foto dari d:\PCD\TBP-PCD\data\foto


# 2. Algoritma Stego

## A. LSB Standar

In [3]:
AKHIR_LSB = "#####"

def teks_ke_biner_lsb(teks, pakai_akhir=True):
    s = teks + AKHIR_LSB if pakai_akhir else teks
    return "".join(f"{b:08b}" for b in s.encode("utf-8"))

def sisipkan_lsb(img, teks):
    bits = teks_ke_biner_lsb(teks, pakai_akhir=True)
    flat = img.copy().flatten()
    if len(bits) > len(flat):
        raise ValueError("Pesan terlalu panjang untuk kapasitas LSB standar.")
    for i, b in enumerate(bits):
        flat[i] = (flat[i] & 0xFE) | int(b)
    return flat.reshape(img.shape)

def ekstrak_bit_lsb(stego_img, target_bit_len):
    flat = stego_img.flatten()
    return "".join(str(flat[i] & 1) for i in range(target_bit_len))

hasil_lsb = []
print("=== PROSES PENYISIPAN: LSB STANDAR ===")
for d in data:
    stego = sisipkan_lsb(d["img"], d["pesan"])
    path_out = LSB_DIR / f"{d['nama']}_stego.png"
    cv2.imwrite(str(path_out), stego)
    hasil_lsb.append({"nama": d["nama"], "pesan": d["pesan"], "cover": d["img"], "stego": stego})
    print(f"Tersimpan: {path_out.name} | Pesan: {d['pesan']}")

=== PROSES PENYISIPAN: LSB STANDAR ===
Tersimpan: BACTERIA-1157929-0001_stego.png | Pesan: RM001
Tersimpan: BACTERIA-4269599-0003_stego.png | Pesan: RM002
Tersimpan: NORMAL-528189-0001_stego.png | Pesan: RM003
Tersimpan: NORMAL-1101909-0001_stego.png | Pesan: RM004


## B. Content Addaptive LSB

In [4]:
AKHIR_CA = "#####"

def get_adaptive_bits(p1, p2):
    ref1 = int(p1) & 0b11111000
    ref2 = int(p2) & 0b11111000
    diff = abs(ref1 - ref2)
    if diff <= 15:
        return 1
    elif diff <= 45:
        return 2
    return 3

def sisipkan_lsb_ca(img, teks):
    bits = "".join(f"{b:08b}" for b in (teks + AKHIR_CA).encode("utf-8"))
    flat = img.copy().flatten()
    bit_idx, total = 0, len(bits)
    for i in range(0, len(flat) - 1, 2):
        if bit_idx >= total:
            break
        k = get_adaptive_bits(flat[i], flat[i + 1])
        chunk = bits[bit_idx : bit_idx + k].ljust(k, "0")
        mask = (0xFF << k) & 0xFF
        flat[i + 1] = (flat[i + 1] & mask) | int(chunk, 2)
        bit_idx += k
    if bit_idx < total:
        raise ValueError("Kapasitas citra tidak mencukupi untuk LSB CA.")
    return flat.reshape(img.shape)

def ekstrak_bit_lsb_ca(stego_img, target_bit_len):
    flat = stego_img.flatten()
    extracted = []
    cur_len = 0
    for i in range(0, len(flat) - 1, 2):
        if cur_len >= target_bit_len:
            break
        k = get_adaptive_bits(flat[i], flat[i + 1])
        val = flat[i + 1] & ((1 << k) - 1)
        extracted.append(format(val, f"0{k}b"))
        cur_len += k
    return "".join(extracted)[:target_bit_len]

hasil_lsb_ca = []
print("=== PROSES PENYISIPAN: CONTENT-ADAPTIVE LSB ===")
for d in data:
    stego = sisipkan_lsb_ca(d["img"], d["pesan"])
    path_out = LSB_CA_DIR / f"{d['nama']}_stego.png"
    cv2.imwrite(str(path_out), stego)
    hasil_lsb_ca.append({"nama": d["nama"], "pesan": d["pesan"], "cover": d["img"], "stego": stego})
    print(f"Tersimpan: {path_out.name} | Pesan: {d['pesan']}")

=== PROSES PENYISIPAN: CONTENT-ADAPTIVE LSB ===
Tersimpan: BACTERIA-1157929-0001_stego.png | Pesan: RM001
Tersimpan: BACTERIA-4269599-0003_stego.png | Pesan: RM002
Tersimpan: NORMAL-528189-0001_stego.png | Pesan: RM003
Tersimpan: NORMAL-1101909-0001_stego.png | Pesan: RM004


## C. DCT

In [5]:
QF_DCT = 90
AKHIR_DCT = "#####"
POSISI_ZIGZAG = [4, 5, 6, 7]
QDASAR = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61], [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56], [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77], [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101], [72, 92, 95, 98, 112, 100, 103, 99]], dtype=np.float32)

def tabel_kuantisasi(qf):
    skala = 5000 / qf if qf < 50 else 200 - 2 * qf
    return np.clip(np.floor((QDASAR * skala + 50) / 100), 1, 255).astype(np.float32)

def urutan_zigzag(n=8):
    p = [(r, c) for r in range(n) for c in range(n)]
    p.sort(key=lambda x: (x[0] + x[1], x[0] if (x[0] + x[1]) % 2 == 1 else x[1]))
    return p

def teks_ke_basis3(teks):
    bits = "".join(f"{b:08b}" for b in (teks + AKHIR_DCT).encode("utf-8"))
    bits += "0" * (-len(bits) % 6)
    digit = []
    for i in range(0, len(bits), 6):
        n, d = int(bits[i:i + 6], 2), []
        for _ in range(4):
            d.append(n % 3); n //= 3
        digit += d[::-1]
    return digit

def basis3_ke_biner(digit, target_bit_len):
    bits = []
    for i in range(0, len(digit) - 3, 4):
        n = 0
        for d in digit[i:i + 4]:
            n = n * 3 + int(d)
        bits.append(format(min(n, 63), "06b"))
    return "".join(bits)[:target_bit_len]

def sesuaikan(c, s):
    for k in sorted(range(c - 2, c + 3), key=lambda k: abs(k - c)):
        if k % 3 == s:
            return k

BATAS_BLOK = 16

def blok_aman(blok):
    # Lewati blok terlalu gelap/terang: perubahan koefisien di sana ter-clip ke 0/255
    # sehingga digit yang disisipkan hilang. Rata-rata blok (DC) tidak diubah oleh
    # penyisipan, jadi aturan ini sama saat sisip maupun ekstrak.
    m = float(blok.mean())
    return BATAS_BLOK <= m <= 255 - BATAS_BLOK

def sisipkan_dct(img, teks, qf=90):
    h, w = img.shape
    digit = teks_ke_basis3(teks)
    Q = tabel_kuantisasi(qf)
    zz = urutan_zigzag()
    posisi = [zz[i] for i in POSISI_ZIGZAG]
    out, k = img.copy(), 0
    for r in range(0, h, 8):
        for c in range(0, w, 8):
            if k >= len(digit):
                return out
            if not blok_aman(out[r:r+8, c:c+8]):
                continue
            q = np.round(cv2.dct(out[r:r+8, c:c+8].astype(np.float32) - 128) / Q).astype(np.int32)
            for (i, j) in posisi:
                q[i, j] = sesuaikan(int(q[i, j]), digit[k])
                k += 1
            rec = cv2.idct(q.astype(np.float32) * Q) + 128
            out[r:r+8, c:c+8] = np.clip(np.round(rec), 0, 255).astype(np.uint8)
    return out

def ekstrak_bit_dct(stego_img, target_bit_len, qf=90):
    h, w = stego_img.shape
    jml_grup_6bit = (target_bit_len + 5) // 6
    target_digit = jml_grup_6bit * 4
    Q = tabel_kuantisasi(qf)
    zz = urutan_zigzag()
    posisi = [zz[i] for i in POSISI_ZIGZAG]
    digit = []
    for r in range(0, h, 8):
        for c in range(0, w, 8):
            if len(digit) >= target_digit:
                return basis3_ke_biner(digit, target_bit_len)
            if not blok_aman(stego_img[r:r+8, c:c+8]):
                continue
            q = np.round(cv2.dct(stego_img[r:r+8, c:c+8].astype(np.float32) - 128) / Q).astype(np.int32)
            for (i, j) in posisi:
                digit.append(int(q[i, j]) % 3)
    return basis3_ke_biner(digit, target_bit_len)

hasil_dct = []
print("=== PROSES PENYISIPAN: DCT BASIS-3 ===")
for d in data:
    stego = sisipkan_dct(d["img"], d["pesan"], QF_DCT)
    path_out = DCT_DIR / f"{d['nama']}_stego.png"
    cv2.imwrite(str(path_out), stego)
    hasil_dct.append({"nama": d["nama"], "pesan": d["pesan"], "cover": d["img"], "stego": stego})
    print(f"Tersimpan: {path_out.name} | Pesan: {d['pesan']}")

=== PROSES PENYISIPAN: DCT BASIS-3 ===
Tersimpan: BACTERIA-1157929-0001_stego.png | Pesan: RM001
Tersimpan: BACTERIA-4269599-0003_stego.png | Pesan: RM002
Tersimpan: NORMAL-528189-0001_stego.png | Pesan: RM003
Tersimpan: NORMAL-1101909-0001_stego.png | Pesan: RM004


# 3. Evaluasi PSNR & BER Sebelum Kompresi

In [6]:
def hitung_ber(teks_asli, bit_ekstrak):
    bit_asli = "".join(f"{b:08b}" for b in teks_asli.encode("utf-8"))
    salah = sum(b1 != b2 for b1, b2 in zip(bit_asli, bit_ekstrak))
    return (salah / len(bit_asli)) * 100.0

rekap_sebelum = []

for i in range(len(data)):
    nama  = data[i]["nama"]
    pesan = data[i]["pesan"]
    cover = data[i]["img"]
    n_bit = len(pesan.encode("utf-8")) * 8

    # 1. LSB Standar
    stego_1 = hasil_lsb[i]["stego"]
    psnr_1  = psnr(cover, stego_1, data_range=255)
    ber_1   = hitung_ber(pesan, ekstrak_bit_lsb(stego_1, n_bit))

    # 2. Content-Adaptive LSB
    stego_2 = hasil_lsb_ca[i]["stego"]
    psnr_2  = psnr(cover, stego_2, data_range=255)
    ber_2   = hitung_ber(pesan, ekstrak_bit_lsb_ca(stego_2, n_bit))

    # 3. DCT Basis-3
    stego_3 = hasil_dct[i]["stego"]
    psnr_3  = psnr(cover, stego_3, data_range=255)
    ber_3   = hitung_ber(pesan, ekstrak_bit_dct(stego_3, n_bit, QF_DCT))

    rekap_sebelum.append({
        "Nama Foto": nama,
        "PSNR LSB (dB)": round(psnr_1, 2), "BER LSB (%)": round(ber_1, 2),
        "PSNR LSB-CA (dB)": round(psnr_2, 2), "BER LSB-CA (%)": round(ber_2, 2),
        "PSNR DCT (dB)": round(psnr_3, 2), "BER DCT (%)": round(ber_3, 2),
    })

df_sebelum = pd.DataFrame(rekap_sebelum)
print("=== TABEL EVALUASI SEBELUM KOMPRESI JPEG ===")
print(df_sebelum.to_string(index=False))
print("\nRATA-RATA SEBELUM KOMPRESI:")
print(df_sebelum.mean(numeric_only=True).round(2).to_string())

=== TABEL EVALUASI SEBELUM KOMPRESI JPEG ===
            Nama Foto  PSNR LSB (dB)  BER LSB (%)  PSNR LSB-CA (dB)  BER LSB-CA (%)  PSNR DCT (dB)  BER DCT (%)
BACTERIA-1157929-0001          85.98          0.0             85.60             0.0          68.58          0.0
BACTERIA-4269599-0003          85.78          0.0             86.30             0.0          70.72          0.0
   NORMAL-528189-0001          86.08          0.0             81.94             0.0          68.58          5.0
  NORMAL-1101909-0001          87.69          0.0             87.69             0.0          78.51         30.0

RATA-RATA SEBELUM KOMPRESI:
PSNR LSB (dB)       86.38
BER LSB (%)          0.00
PSNR LSB-CA (dB)    85.38
BER LSB-CA (%)       0.00
PSNR DCT (dB)       71.60
BER DCT (%)          8.75


# 4. Kompresi JPEG Berbagai Tingkat Kualitas

In [7]:
TINGKAT_QF = [90, 75, 50]
hasil_kompresi = {qf: {"lsb": [], "lsb_ca": [], "dct": []} for qf in TINGKAT_QF}

print("=== PROSES KOMPRESI JPEG (QF = 90, 75, 50) ===")
for qf in TINGKAT_QF:
    sub_dir = KOMPRES_DIR / f"QF_{qf}"
    sub_dir.mkdir(parents=True, exist_ok=True)
    for i in range(len(data)):
        nama = data[i]["nama"]
        for metode, sumber in [("lsb", hasil_lsb), ("lsb_ca", hasil_lsb_ca), ("dct", hasil_dct)]:
            jpg_path = sub_dir / f"{nama}_{metode}_QF{qf}.jpg"
            cv2.imwrite(str(jpg_path), sumber[i]["stego"], [cv2.IMWRITE_JPEG_QUALITY, qf])
            img_jpg = cv2.imread(str(jpg_path), cv2.IMREAD_GRAYSCALE)
            hasil_kompresi[qf][metode].append(img_jpg)
    print(f"Kompresi JPEG QF={qf}% selesai disimpan ke {sub_dir}")

=== PROSES KOMPRESI JPEG (QF = 90, 75, 50) ===
Kompresi JPEG QF=90% selesai disimpan ke d:\PCD\TBP-PCD\hasil\kompresi_jpeg\QF_90
Kompresi JPEG QF=75% selesai disimpan ke d:\PCD\TBP-PCD\hasil\kompresi_jpeg\QF_75
Kompresi JPEG QF=50% selesai disimpan ke d:\PCD\TBP-PCD\hasil\kompresi_jpeg\QF_50


# 5. Evaluasi PSNR & BER Setelah Kompresi JPEG

In [8]:
rekap_sesudah = []

for qf in TINGKAT_QF:
    for i in range(len(data)):
        nama  = data[i]["nama"]
        pesan = data[i]["pesan"]
        cover = data[i]["img"]
        n_bit = len(pesan.encode("utf-8")) * 8

        # 1. LSB Standar pasca-JPEG
        jpg_1  = hasil_kompresi[qf]["lsb"][i]
        psnr_1 = psnr(cover, jpg_1, data_range=255)
        ber_1  = hitung_ber(pesan, ekstrak_bit_lsb(jpg_1, n_bit))

        # 2. Content-Adaptive LSB pasca-JPEG
        jpg_2  = hasil_kompresi[qf]["lsb_ca"][i]
        psnr_2 = psnr(cover, jpg_2, data_range=255)
        ber_2  = hitung_ber(pesan, ekstrak_bit_lsb_ca(jpg_2, n_bit))

        # 3. DCT Basis-3 pasca-JPEG
        jpg_3  = hasil_kompresi[qf]["dct"][i]
        psnr_3 = psnr(cover, jpg_3, data_range=255)
        ber_3  = hitung_ber(pesan, ekstrak_bit_dct(jpg_3, n_bit, QF_DCT))

        rekap_sesudah.append({
            "QF JPEG": f"{qf}%",
            "Nama Foto": nama,
            "PSNR LSB": round(psnr_1, 2), "BER LSB (%)": round(ber_1, 2),
            "PSNR LSB-CA": round(psnr_2, 2), "BER LSB-CA (%)": round(ber_2, 2),
            "PSNR DCT": round(psnr_3, 2), "BER DCT (%)": round(ber_3, 2),
        })

df_sesudah = pd.DataFrame(rekap_sesudah)
print("=== TABEL REKAP PER FOTO SETELAH KOMPRESI JPEG ===")
print(df_sesudah.to_string(index=False))

print("\n=== TABEL RATA-RATA PER TINGKAT KOMPRESI JPEG ===")
df_rata = df_sesudah.groupby("QF JPEG", sort=False).mean(numeric_only=True).round(2)
print(df_rata.to_string())

=== TABEL REKAP PER FOTO SETELAH KOMPRESI JPEG ===
QF JPEG             Nama Foto  PSNR LSB  BER LSB (%)  PSNR LSB-CA  BER LSB-CA (%)  PSNR DCT  BER DCT (%)
    90% BACTERIA-1157929-0001     43.32         42.5        43.32            37.5     43.32          0.0
    90% BACTERIA-4269599-0003     45.58         65.0        45.58            30.0     45.57          0.0
    90%    NORMAL-528189-0001     42.71         47.5        42.71            57.5     42.71          5.0
    90%   NORMAL-1101909-0001     43.34         35.0        43.34            35.0     43.34         27.5
    75% BACTERIA-1157929-0001     40.70         40.0        40.70            42.5     40.70         47.5
    75% BACTERIA-4269599-0003     42.36         50.0        42.36            45.0     42.36         42.5
    75%    NORMAL-528189-0001     40.52         37.5        40.52            52.5     40.52         50.0
    75%   NORMAL-1101909-0001     40.81         35.0        40.81            35.0     40.81         35.0
    